# Chapter 7 · Training and Data

Training with method: three datasets, contamination, optimizers, the learning rate, checkpoints and a training log. PyTorch on the CPU. The whole notebook takes about ten minutes.

The full explanation is in the PDF chapter. This notebook is the lab.

## 1. Data, configurations and batches

In Colab, the first cell downloads the novel if it is not there.

In [1]:
"""Chapter 7: training with method. PyTorch on CPU."""
import json
import math
import platform
import random
import time
from pathlib import Path

import torch
from torch import nn
from torch.nn import functional as F

ROOT = Path(".")
TEXT = Path("material/quixote.txt")
if not TEXT.exists():  # in Colab, download it from the repository
    import urllib.request
    TEXT.parent.mkdir(exist_ok=True)
    urllib.request.urlretrieve("https://raw.githubusercontent.com/HernanDiaz/language-models/main/material/quixote.txt", TEXT)
OUTPUTS = Path("outputs")
SEED = 1337
LAB = dict(context=32, dimension=32, heads=2, layers=2, batch=32)
FINAL = dict(context=64, dimension=64, heads=4, layers=4, batch=32)


# --- Data ------------------------------------------------------------------

def prepare(text):
    symbols = sorted(set(text))
    to_id = {c: i for i, c in enumerate(symbols)}
    data = torch.tensor([to_id[c] for c in text], dtype=torch.long)
    a, b = int(0.8 * len(data)), int(0.9 * len(data))
    parts = {"training": data[:a], "validation": data[a:b],
             "test": data[b:]}
    return symbols, to_id, parts


def overlap(text_a, text_b, n=50, samples=2000, seed=0):
    """Count how many n-character pieces of text_a appear in text_b."""
    rng = random.Random(seed)
    starts = [rng.randrange(len(text_a) - n) for _ in range(samples)]
    return sum(text_a[i:i + n] in text_b for i in starts)


def get_batch(data, config, generator):
    T, B = config["context"], config["batch"]
    starts = torch.randint(len(data) - T - 1, (B,), generator=generator)
    x = torch.stack([data[i:i + T] for i in starts])
    y = torch.stack([data[i + 1:i + T + 1] for i in starts])
    return x, y

In [2]:
torch.set_num_threads(4)
text = TEXT.read_text(encoding="utf-8")
symbols, to_id, parts = prepare(text)
print({name: len(data) for name, data in parts.items()})
cut_a, cut_b = int(0.8 * len(text)), int(0.9 * len(text))
for n in (5, 10, 20, 30, 50, 100):
    repeated = overlap(text[cut_b:], text[:cut_a], n)
    print("Test pieces of %d characters found in training: %d of 2000" % (n, repeated))

{'training': 1744341, 'validation': 218043, 'test': 218043}
Test pieces of 5 characters found in training: 1904 of 2000
Test pieces of 10 characters found in training: 896 of 2000
Test pieces of 20 characters found in training: 55 of 2000
Test pieces of 30 characters found in training: 6 of 2000
Test pieces of 50 characters found in training: 0 of 2000
Test pieces of 100 characters found in training: 0 of 2000


## 2. The model of chapter 6, with its size as a parameter, and the two measures of the loss

In [3]:
# --- Model: the pieces of chapter 6, with their size as a parameter -------

class Head(nn.Module):
    def __init__(self, config):
        super().__init__()
        C, T = config["dimension"], config["context"]
        size = C // config["heads"]
        self.query = nn.Linear(C, size, bias=False)
        self.key = nn.Linear(C, size, bias=False)
        self.value = nn.Linear(C, size, bias=False)
        self.register_buffer("mask", torch.tril(torch.ones(T, T)))

    def forward(self, x):
        T = x.shape[1]
        q, k, v = self.query(x), self.key(x), self.value(x)
        scores = q @ k.transpose(1, 2) / math.sqrt(k.shape[-1])
        future = self.mask[:T, :T] == 0
        scores = scores.masked_fill(future, float("-inf"))
        return F.softmax(scores, dim=-1) @ v


class Block(nn.Module):
    def __init__(self, config):
        super().__init__()
        C = config["dimension"]
        self.heads = nn.ModuleList(
            [Head(config) for _ in range(config["heads"])])
        self.projection = nn.Linear(C, C)
        self.mlp = nn.Sequential(nn.Linear(C, 4 * C), nn.ReLU(),
                                 nn.Linear(4 * C, C))
        self.norm1, self.norm2 = nn.LayerNorm(C), nn.LayerNorm(C)

    def forward(self, x):
        h = self.norm1(x)
        attention = torch.cat([head(h) for head in self.heads], dim=-1)
        x = x + self.projection(attention)
        return x + self.mlp(self.norm2(x))


class GPT(nn.Module):
    def __init__(self, vocab_size, config):
        super().__init__()
        C = config["dimension"]
        self.config = config
        self.symbol_embedding = nn.Embedding(vocab_size, C)
        self.position_embedding = nn.Embedding(config["context"], C)
        self.blocks = nn.Sequential(
            *[Block(config) for _ in range(config["layers"])])
        self.norm = nn.LayerNorm(C)
        self.output = nn.Linear(C, vocab_size)

    def forward(self, ids):
        positions = torch.arange(ids.shape[1])
        x = self.symbol_embedding(ids) + self.position_embedding(positions)
        return self.output(self.norm(self.blocks(x)))


def mean_loss(model, x, y):
    logits = model(x)
    return F.cross_entropy(logits.view(-1, logits.shape[-1]), y.view(-1))


@torch.no_grad()
def evaluate(model, data, batches=20):
    """Quick measure: the mean loss on 20 batches, always the same ones."""
    generator = torch.Generator().manual_seed(0)
    losses = [mean_loss(model, *get_batch(data, model.config, generator))
              for _ in range(batches)]
    return sum(loss.item() for loss in losses) / batches


@torch.no_grad()
def evaluate_all(model, data, group=256):
    """Full measure: the mean loss on the whole block, piece by piece."""
    T = model.config["context"]
    n = (len(data) - 1) // T
    x, y = data[:n * T].view(n, T), data[1:n * T + 1].view(n, T)
    total = 0.0
    for i in range(0, n, group):
        total += mean_loss(model, x[i:i + group],
                           y[i:i + group]).item() * len(x[i:i + group])
    return total / n

## 3. Optimizers and learning-rate schedule

In [4]:
# --- Optimizers and learning rate --------------------------------------------

def make_optimizer(model, name, rate):
    if name == "sgd":
        return torch.optim.SGD(model.parameters(), lr=rate)
    if name == "momentum":
        return torch.optim.SGD(model.parameters(), lr=rate, momentum=0.9)
    return torch.optim.AdamW(model.parameters(), lr=rate, weight_decay=0.1)


def rate_at(step, rate, steps, warmup=100, final=0.1):
    """Linear warm-up, then cosine decay down to final * rate."""
    if step < warmup:
        return rate * (step + 1) / warmup
    done = (step - warmup) / max(1, steps - warmup)
    return rate * (final + (1 - final) * 0.5 * (1 + math.cos(math.pi * done)))

In [5]:
for step in (0, 50, 100, 1550, 2999):
    print(step, rate_at(step, 3e-3, 3000))

0 3e-05
50 0.00153
100 0.003
1550 0.0016500000000000002
2999 0.0003000007921500975


## 4. Train, save and load

In [6]:
# --- Train, save and resume --------------------------------------------------

def new_state(vocab_size, config, optimizer, rate):
    torch.manual_seed(SEED)
    model = GPT(vocab_size, config)
    return {"model": model,
            "optimizer": make_optimizer(model, optimizer, rate),
            "generator": torch.Generator().manual_seed(SEED),
            "step": 0, "history": []}


def train(state, parts, until, rate, total_steps, schedule=False,
          interval=250, folder=None):
    model, optimizer = state["model"], state["optimizer"]
    history = state["history"]
    best = min([h[2] for h in history], default=math.inf)
    while state["step"] <= until:
        step = state["step"]
        already_measured = history and history[-1][0] == step
        if step % interval == 0 and not already_measured:
            measure = (step, evaluate(model, parts["training"]),
                       evaluate(model, parts["validation"]))
            history.append(measure)
            if folder and measure[2] < best:
                best = measure[2]
                save(state, folder / "best.pt")
        if step == until:
            break
        if schedule:
            for group in optimizer.param_groups:
                group["lr"] = rate_at(step, rate, total_steps)
        x, y = get_batch(parts["training"], model.config, state["generator"])
        loss = mean_loss(model, x, y)
        optimizer.zero_grad(set_to_none=True)
        loss.backward()
        optimizer.step()
        state["step"] += 1
    return state


def save(state, path):
    path.parent.mkdir(parents=True, exist_ok=True)
    torch.save({"model": state["model"].state_dict(),
                "config": state["model"].config,
                "optimizer": state["optimizer"].state_dict(),
                "generator": state["generator"].get_state(),
                "step": state["step"],
                "history": state["history"]}, path)


def load(path, vocab_size, optimizer, rate):
    checkpoint = torch.load(path)
    state = new_state(vocab_size, checkpoint["config"], optimizer, rate)
    state["model"].load_state_dict(checkpoint["model"])
    state["optimizer"].load_state_dict(checkpoint["optimizer"])
    state["generator"].set_state(checkpoint["generator"])
    state["step"], state["history"] = checkpoint["step"], checkpoint["history"]
    return state

## 5. The lab: optimizers and rates

Nine trainings of 1000 steps with the small model. It takes about three minutes.

In [7]:
# --- The chapter's experiments -----------------------------------------------

def lab(symbols, parts, steps=1000):
    results = {}
    trials = [("sgd", 0.1), ("sgd", 1.0), ("momentum", 0.1),
              ("adamw", 3e-4), ("adamw", 1e-3), ("adamw", 3e-3),
              ("adamw", 1e-2), ("adamw", 3e-2), ("adamw", 1e-1)]
    for name, rate in trials:
        state = new_state(len(symbols), LAB, name, rate)
        train(state, parts, steps, rate, steps, interval=100)
        results[f"{name} {rate:g}"] = state["history"]
        print("%-14s final validation %.3f" % (f"{name} {rate:g}",
                                               state["history"][-1][2]))
    OUTPUTS.mkdir(parents=True, exist_ok=True)
    (OUTPUTS / "lab.json").write_text(
        json.dumps(results, indent=2), encoding="utf-8")
    return results

In [8]:
results = lab(symbols, parts)

sgd 0.1        final validation 2.550
sgd 1          final validation 2.169
momentum 0.1   final validation 2.160
adamw 0.0003   final validation 2.460
adamw 0.001    final validation 2.179
adamw 0.003    final validation 1.928
adamw 0.01     final validation 1.886
adamw 0.03     final validation 1.969
adamw 0.1      final validation 2.427


## 6. Interrupt and resume

In [9]:
def resume(symbols, parts, steps=600):
    straight = new_state(len(symbols), LAB, "adamw", 1e-2)
    train(straight, parts, steps, 1e-2, steps, schedule=True,
          interval=100)
    half = new_state(len(symbols), LAB, "adamw", 1e-2)
    train(half, parts, steps // 2, 1e-2, steps, schedule=True,
          interval=100)
    save(half, OUTPUTS / "halfway.pt")
    del half  # we pretend that the program stops here
    resumed = load(OUTPUTS / "halfway.pt", len(symbols), "adamw", 1e-2)
    train(resumed, parts, steps, 1e-2, steps, schedule=True,
          interval=100)
    return straight["history"][-1], resumed["history"][-1]

In [10]:
print(resume(symbols, parts))

((600, 1.992661678791046, 1.987269300222397), (600, 1.992661678791046, 1.987269300222397))


## 7. The final experiment

The model of chapter 6 with a schedule and checkpoints. It takes about five and a half minutes.

In [11]:
def final_experiment(symbols, parts, steps=3000, rate=3e-3):
    state = new_state(len(symbols), FINAL, "adamw", rate)
    start = time.perf_counter()
    train(state, parts, steps, rate, steps, schedule=True,
          folder=OUTPUTS)
    seconds = time.perf_counter() - start
    best = load(OUTPUTS / "best.pt", len(symbols), "adamw", rate)
    model = best["model"]
    log = {
        "date": time.strftime("%Y-%m-%d %H:%M"),
        "python": platform.python_version(), "torch": torch.__version__,
        "threads": torch.get_num_threads(), "seed": SEED,
        "split": "80/10/10 in consecutive blocks", "config": FINAL,
        "parameters": sum(p.numel() for p in model.parameters()),
        "optimizer": "adamw", "brake": 0.1, "rate": rate,
        "schedule": "100-step warm-up, then cosine down to 10%",
        "steps": steps, "seconds": round(seconds),
        "measures": "20 batches every 250 steps; full blocks at the end",
        "history": state["history"], "best_step": best["step"],
        "validation": evaluate_all(model, parts["validation"]),
        "test": evaluate_all(model, parts["test"])}  # only once
    (OUTPUTS / "log.json").write_text(json.dumps(log, indent=2), encoding="utf-8")
    final = {"model": model.state_dict(), "config": FINAL, "symbols": symbols}
    torch.save(final, OUTPUTS / "final_model.pt")
    torch.save(final, ROOT / "material" / "gpt_quixote.pt")  # for chapters 8-10
    return log

In [12]:
log = final_experiment(symbols, parts)
print("Best step:", log["best_step"], "· validation: %.3f" % log["validation"],
      "· test: %.3f" % log["test"])
print((OUTPUTS / "log.json").read_text(encoding="utf-8")[:900])

Best step: 3000 · validation: 1.480 · test: 1.478
{
  "date": "2026-10-04 15:47",
  "python": "3.12.14",
  "torch": "2.8.0+cpu",
  "threads": 4,
  "seed": 1337,
  "split": "80/10/10 in consecutive blocks",
  "config": {
    "context": 64,
    "dimension": 64,
    "heads": 4,
    "layers": 4,
    "batch": 32
  },
  "parameters": 215002,
  "optimizer": "adamw",
  "brake": 0.1,
  "rate": 0.003,
  "schedule": "100-step warm-up, then cosine down to 10%",
  "steps": 3000,
  "seconds": 322,
  "measures": "20 batches every 250 steps; full blocks at the end",
  "history": [
    [
      0,
      4.593560338020325,
      4.592600321769714
    ],
    [
      250,
      2.248047077655792,
      2.2609360456466674
    ],
    [
      500,
      1.9195975303649901,
      1.9394337058067321
    ],
    [
      750,
      1.7802375912666322,
      1.800136148929596
    ],
    [
      1000,
      1.6861225605010985,
      1.709775447845459
    ],
    [
   


## Exercises

1. Explain in your own words what each of the three datasets is for. What would happen if we chose the learning rate by looking at the test loss?

2. Why do we cut the text into consecutive blocks instead of distributing sentences or fragments at random?

3. According to the chapter's table, finding a 10-character piece of the test set in the training set does not indicate contamination, but finding a 100-character one would. Why?

4. With the lab model, each batch has 32 fragments of 32 characters. How many predictions does it make in 1000 steps? How many epochs is that?

5. Apply two steps of momentum by hand, with v=0 at the start, rate 0.1 and gradients g_1=1 and g_2=1. How much does the parameter move in total? Compare it with two steps of plain gradient descent.

6. One parameter always has a gradient of 0.001, and another, of 10. How much does each one move in a step of gradient descent with rate 0.003? And, approximately, with Adam, once m and s have settled?

7. With the program's function `rate_at`, compute the rate at steps 0, 50, 100, 1550 and 2999 of the final training.

8. Open `outputs/lab.json` and find, for each AdamW rate, the validation loss at step 300. Which rate wins at 300 steps? And at 1000? From which step on does the second one always win? What does this teach you about the length of a sweep?

9. Check what happens if we forget the state of the optimizer. Remove from `load` the line that loads it, run only the resume test (in the notebook, its cell) and compare the two losses. Why do they no longer match? Why is the difference small?

10. Put `load` back as it was. Then modify `save` so that it does not save the state of the generator, remove from `load` the line with `set_state` and repeat the resume test. Explain the result.

11. Sweep with the final model. Adapt `lab` so that it uses the `FINAL` configuration and only AdamW with the rates 0.001, 0.003 and 0.01, and compare their validation loss after 1000 steps. It takes about six minutes. Is 0.003 still a good choice? Does the large model prefer a smaller rate than the lab model?

12. Training log. Open `outputs/log.json` and write a short report, about ten lines long, with the configuration, the evolution of the losses, the best checkpoint and the final losses. Say which decisions were made with the validation set and when the test set was used.

13. Introduce contamination on purpose: add the first 50,000 characters of the test set to the end of the training text and measure the overlap again with 50-character pieces. What would you expect to happen to the test loss if you trained like that?

## References

Inspired by [nanoGPT](https://github.com/karpathy/nanoGPT) and by the lecture [Let's reproduce GPT-2](https://www.youtube.com/watch?v=l8pRSuU81PU), by Andrej Karpathy. Text of *Don Quixote*, by Miguel de Cervantes, in John Ormsby's English translation, in the public domain. The code and the experiments are original.